- Building the SFT dataset for our DPO vs GRPO project 

    - EleutherAI/hendrycks_math: MATH, 7 subjects(train split only used for training)
    - HuggingFaceH4/MATH-500: 500 problems from the MATH test split (this is our main evaluation benchmark, we never train on this)
    - meta-math/MetaMathQA: augmentations of MATH train problems (we keep MATH-derived types only)

    - Outputs:
        - sft_all: cleaned SFT data with columns problem, solution, answer, subject, level, source
        - math_val: approx 500 held-out MATH train problems for checkpoint
        - math500: eval set

- NOTE:
    - NOTHING FROM THE MATH TEST SPLIT GOES INTO TRAINING

# 1.Loading the raw datasets

In [8]:
import re
import random
from collections import Counter, defaultdict

import pandas as pd
from datasets import load_dataset, concatenate_datasets

SEED = 42          # fixed seed so the validation split is reproducible
VAL_SIZE = 500     # target size of the held-out validation split

# MetaMathQA types to drop entirely (see section 6 for why)
#   MATH_SV    -> dropped: reasoning often contradicts the label, "unknown variable X" format
#   MATH_FOBAR -> add here if you decide to drop it after inspecting section 6
# MATH_FOBAR only 64 of its 13k rows have \boxed{label}
DROP_TYPES = {"MATH_SV", "MATH_FOBAR"} 

In [9]:
SUBJECTS = ["algebra", "counting_and_probability", "geometry", "intermediate_algebra", "number_theory", "prealgebra", "precalculus"]

# MATH train will combine all 7 subjects 
math_train = concatenate_datasets(
    [load_dataset("EleutherAI/hendrycks_math", s, split = 'train') for s in SUBJECTS]
)

# MATH-500 evaluation set 
math500 = load_dataset("HuggingFaceH4/MATH-500", split = 'test')

# MetaMathQA is another dataset wwe can use for extra SFT but we only filter to the MATH types 
metamath = load_dataset("meta-math/MetaMathQA", split = 'train')
metamath_math = metamath.filter(lambda x: x['type'].startswith("MATH_"))

print(len(math_train),len(math500), len(metamath_math))
print(math_train[0])

7500 500 155000
{'problem': 'Let \\[f(x) = \\left\\{\n\\begin{array}{cl} ax+3, &\\text{ if }x>2, \\\\\nx-5 &\\text{ if } -2 \\le x \\le 2, \\\\\n2x-b &\\text{ if } x <-2.\n\\end{array}\n\\right.\\]Find $a+b$ if the piecewise function is continuous (which means that its graph can be drawn without lifting your pencil from the paper).', 'level': 'Level 5', 'type': 'Algebra', 'solution': 'For the piecewise function to be continuous, the cases must "meet" at $2$ and $-2$. For example, $ax+3$ and $x-5$ must be equal when $x=2$. This implies $a(2)+3=2-5$, which we solve to get $2a=-6 \\Rightarrow a=-3$. Similarly, $x-5$ and $2x-b$ must be equal when $x=-2$. Substituting, we get $-2-5=2(-2)-b$, which implies $b=3$. So $a+b=-3+3=\\boxed{0}$.'}


In [10]:
from pprint import pprint
pprint(math_train[0])

{'level': 'Level 5',
 'problem': 'Let \\[f(x) = \\left\\{\n'
            '\\begin{array}{cl} ax+3, &\\text{ if }x>2, \\\\\n'
            'x-5 &\\text{ if } -2 \\le x \\le 2, \\\\\n'
            '2x-b &\\text{ if } x <-2.\n'
            '\\end{array}\n'
            '\\right.\\]Find $a+b$ if the piecewise function is continuous '
            '(which means that its graph can be drawn without lifting your '
            'pencil from the paper).',
 'solution': 'For the piecewise function to be continuous, the cases must '
             '"meet" at $2$ and $-2$. For example, $ax+3$ and $x-5$ must be '
             'equal when $x=2$. This implies $a(2)+3=2-5$, which we solve to '
             'get $2a=-6 \\Rightarrow a=-3$. Similarly, $x-5$ and $2x-b$ must '
             'be equal when $x=-2$. Substituting, we get $-2-5=2(-2)-b$, which '
             'implies $b=3$. So $a+b=-3+3=\\boxed{0}$.',
 'type': 'Algebra'}


# 2. Answer extraction 

- extracting the \boxed{} format from the dataset which is a json format.
- This is not the grader just extracting the boxed answer

In [11]:
def get_last_boxed(text):
    # Find the last \boxed or \fbox, whichever comes later
    idx_boxed = text.rfind(r"\boxed")
    idx_fbox = text.rfind(r"\fbox")
    if idx_boxed == -1 and idx_fbox == -1:
        return None
    if idx_boxed >= idx_fbox:
        current_idx = idx_boxed + len(r"\boxed")
    else:
        current_idx = idx_fbox + len(r"\fbox")

    # Skip whitespace
    while current_idx < len(text) and text[current_idx].isspace():
        current_idx += 1

    # No brace: handle "\boxed 5" by taking the next token
    if current_idx >= len(text):
        return None
    if text[current_idx] != "{":
        token = text[current_idx:].split()[0].rstrip("$.")
        return token or None

    # Parse nested braces
    current_idx += 1
    brace_depth = 1
    content_start_idx = current_idx
    while current_idx < len(text) and brace_depth > 0:
        if text[current_idx] == "{":
            brace_depth += 1
        elif text[current_idx] == "}":
            brace_depth -= 1
        current_idx += 1

    if brace_depth != 0:
        return None

    answer = text[content_start_idx:current_idx - 1].strip()
    return answer or None

In [12]:
tests = {
    r"so $a+b=\boxed{0}$.": "0",
    r"\boxed{\frac{1}{2}}": r"\frac{1}{2}",
    r"\boxed{\{1,2\}}": r"\{1,2\}",
    r"first \boxed{1} then \boxed{2}": "2",
    r"answer is \fbox{7}": "7",
    r"\boxed 5.": "5",
    r"\boxed{}": None,
    r"\boxed{\frac{1}{2": None,   # truncated output
    "no answer here": None,
}
for text, expected in tests.items():
    got = get_last_boxed(text)
    status = "OK " if got == expected else "FAIL"
    print(f"{status} {text!r} -> {got!r}")

OK  'so $a+b=\\boxed{0}$.' -> '0'
OK  '\\boxed{\\frac{1}{2}}' -> '\\frac{1}{2}'
OK  '\\boxed{\\{1,2\\}}' -> '\\{1,2\\}'
OK  'first \\boxed{1} then \\boxed{2}' -> '2'
OK  'answer is \\fbox{7}' -> '7'
OK  '\\boxed 5.' -> '5'
OK  '\\boxed{}' -> None
OK  '\\boxed{\\frac{1}{2' -> None
OK  'no answer here' -> None


# 3. Clean MATH train: extracting answers and droping fails

In [13]:
# Testting the get_last_boxed function on the math_train dataset
math_train = math_train.map(lambda x: {"answer": get_last_boxed(x["solution"])})
failed = math_train.filter(lambda x: x["answer"] is None) # We should almost have no failed cases, but if we do they get dropped 
print(f"{len(failed) / len(math_train):.2%} of the math_train dataset failed to extract an answer.")

if len(failed):
    print(failed[0]["solution"][-300:])

Map:   0%|          | 0/7500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7500 [00:00<?, ? examples/s]

0.03% of the math_train dataset failed to extract an answer.
Observe that for all $k \in 1< k< n$, since $k$ divides $n!$, $k$ also divides $n!+k$. Therefore, all numbers $a$ in the range $n!+1<a<n!+n$ are composite. Therefore there are $\boxed{}$ primes in that range.


# 4. Validation split
- again here holding out 500 MATH problems for checkpoint section stratified by subject and level, so the mix matches MATH-500. MATH-500 IS NEVER USED FOR TUNING

In [14]:
# Holding out the validation split. We will take 500 problems out of MATH train for picking the checkpoints, so MATH-500 is not touched.

# Group by indicies (subject, level)
groups = defaultdict(list)
for i, ex in enumerate(math_train):
    groups[(ex["type"], ex["level"])].append(i)


# Take the same fraction from every group
rng = random.Random(SEED)
frac = VAL_SIZE / len(math_train)
val_idx = []
for key, idxs in groups.items():
    rng.shuffle(idxs)
    val_idx += idxs[:round(len(idxs) * frac)]

val_set = set(val_idx)
math_val = math_train.select(sorted(val_idx))
math_sft = math_train.select([i for i in range(len(math_train)) if i not in val_set])
print(len(math_sft), len(math_val))

6998 502


In [15]:
print(metamath_math.column_names)
pprint(metamath_math[0])

['type', 'query', 'original_question', 'response']
{'original_question': 'Gracie and Joe are choosing numbers on the complex '
                      'plane. Joe chooses the point $1+2i$. Gracie chooses '
                      "$-1+i$. How far apart are Gracie and Joe's points?",
 'query': 'Gracie and Joe are choosing numbers on the complex plane. Joe '
          'chooses the point $1+2i$. Gracie chooses $-1+i$. How far apart are '
          "Gracie and Joe's points?",
 'response': 'The distance between two points $(x_1,y_1)$ and $(x_2,y_2)$ in '
             'the complex plane is given by the formula '
             '$\\sqrt{(x_2-x_1)^2+(y_2-y_1)^2}$.\n'
             "In this case, Joe's point is $(1,2)$ and Gracie's point is "
             '$(-1,1)$.\n'
             'So the distance between their points is '
             '$\\sqrt{((-1)-(1))^2+((1)-(2))^2}=\\sqrt{(-2)^2+(-1)^2}=\\sqrt{4+1}=\\sqrt{5}$.\n'
             "Therefore, Gracie and Joe's points are $\\boxed{\\sqrt{5}}$ "
       

# 5. MetaMathQA: removing the validation leak 

- MetaMath rewrites MATH train problems (original_question), so any row derived from a validation problem must go or we inflate our validation score.

In [16]:
# we need to remove the validation leak, so using MetaMathQA orginal_question we should find the rows that match with MATH train problems and normalizes so we don't have contamination 
def norm(s):
    return " ".join(s.split())

val_problems = {norm(p) for p in math_val["problem"]}
all_train_problems = {norm(p) for p in math_train["problem"]}

# Sanity check: what fraction of MetaMathQA rows trace back to MATH train?
orig = [norm(q) for q in metamath_math["original_question"]]
match_rate = sum(q in all_train_problems for q in orig) / len(orig)
print(f"Match rate to MATH train: {match_rate:.1%}")

# Drop rows derived from validation problems
before = len(metamath_math)
mm = metamath_math.filter(lambda x: norm(x["original_question"]) not in val_problems)
print(f"Dropped {before - len(mm)} rows derived from validation problems")


Match rate to MATH train: 100.0%
Dropped 10047 rows derived from validation problems


- our leak filter worked we got a 100% match rate and dropped 10,047 rows about 6.5% of MetaMathQA's MATH subset

# 6. MetaMathQA: ansewr check consistency 
- so each response contains a \boxed{} most of the time, and has a trailing "The answer is: X" lebel. We need to check that they agree.
    - loose_norm is data filtering only: never used for grading a models output 

In [17]:
# Matches the trailing "The answer is: ..." line at the very end of a response
ANS_RE = re.compile(r"\n?The answer is:\s*(.*)\s*$")

def loose_norm(s):
    if s is None:
        return None
    s = s.replace("\\$", "").replace("$", "")
    s = s.replace("^\\circ", "").replace("^{\\circ}", "").replace("\\%", "")
    s = re.sub(r"\\text\{(.*?)\}", r"\1", s)
    s = s.replace("\\!", "").replace("\\,", "").replace("\\left", "").replace("\\right", "")
    s = s.replace("dfrac", "frac").replace("tfrac", "frac")
    return "".join(s.split())

def split_response(resp):
    """Split a MetaMathQA response into (body without label line, label)."""
    m = ANS_RE.search(resp)
    tail = m.group(1).strip() if m else None
    body = (resp[:m.start()] if m else resp).rstrip()
    return body, tail

def check(ex):
    body, tail = split_response(ex["response"])
    boxed = get_last_boxed(body)
    return {
        "boxed": boxed,
        "tail": tail,
        "has_boxed": boxed is not None,
        "agree_strict": boxed is not None and tail is not None and norm(boxed) == norm(tail),
        "agree_loose": boxed is not None and tail is not None and loose_norm(boxed) == loose_norm(tail),
    }

chk = mm.map(check)
df = chk.to_pandas()

print("How many rows have a box, by type:")
print(df.groupby("type")["has_boxed"].agg(["sum", "count", "mean"]), "\n")
print("Agreement (rows with a box only):")
print(df[df.has_boxed].groupby("type")[["agree_strict", "agree_loose"]].mean())

How many rows have a box, by type:
                  sum  count      mean
type                                  
MATH_AnsAug     67786  70167  0.966067
MATH_FOBAR         64  13984  0.004577
MATH_Rephrased  44607  46781  0.953528
MATH_SV           102  14021  0.007275 

Agreement (rows with a box only):
                agree_strict  agree_loose
type                                     
MATH_AnsAug         0.938394     0.987534
MATH_FOBAR          1.000000     1.000000
MATH_Rephrased      0.941242     0.990697
MATH_SV             0.549020     0.549020


In [18]:
# Manual inspection of the backward-reasoning types.
# For the paper: inspect 20 of each and count how many have reasoning that contradicts the label.
for t in ["MATH_SV", "MATH_FOBAR"]:
    rows = chk.filter(lambda x: x["type"] == t).shuffle(seed=0).select(range(5))
    print(f"================ {t} ================")
    for ex in rows:
        print(ex["response"][-400:])
        print("---")

================ MATH_SV ================
ble x is 10.
The answer is: 10
---
ber of new employees who can monitor water pollution + Number of new employees who can monitor air pollution - Number of new employees who can monitor both = Total number of new employees
x + 73 - 27 = 131
Let's simplify and solve for x:
x + 73 - 27 = 131
x + 46 = 131
To isolate x, we subtract 46 from both sides of the equation:
x + 46 - 46 = 131 - 46
x = 85
The value of x is 85.
The answer is: 85
---
te value of the difference is 11, we can see that Number 2 cannot be negative. Therefore, Number 2 = 7.
Substituting this back into Equation 2, we can solve for Number 1:
Number 1 = 126 / Number 2
Number 1 = 126 / 7
Number 1 = 18
Now, we can find the sum of the two numbers, which is X:
X = Number 1 + Number 2
X = 18 + 7
X = 25
Therefore, the value of the unknown variable x is 25.
The answer is: 25
---
s is: 92 + 95 + 87 + 89 + X + 95.
We can set up the equation as follows:
(92 + 95 + 87 + 89 + X + 95) / 6 = 93
Le

In [19]:
# Drop whole types decided above (MATH_SV; optionally MATH_FOBAR)
print("Before:", Counter(chk["type"]))
chk = chk.filter(lambda x: x["type"] not in DROP_TYPES)
print("After: ", Counter(chk["type"]))

Before: Counter({'MATH_AnsAug': 70167, 'MATH_Rephrased': 46781, 'MATH_SV': 14021, 'MATH_FOBAR': 13984})
After:  Counter({'MATH_AnsAug': 70167, 'MATH_Rephrased': 46781})


# 7. MetaMathQA: finalize rows

- if we have a box and it agrees with the label (loose) --> keep as is
- if we have no box --> keep and append \boxed{label}
- box disagrees with label, or no answer --> drop

- We remove "The answer is:" line so every solution ends in \boxed{}.+
- The row inherits subject/level from the orignal MATH problem.

In [20]:
# original problem -> (subject, level)
math_lookup = {norm(p): (t, l) for p, t, l in
               zip(math_train["problem"], math_train["type"], math_train["level"])}

def finalize(ex):
    body, tail = split_response(ex["response"])
    boxed = get_last_boxed(body)

    if boxed is not None and tail is not None and loose_norm(boxed) == loose_norm(tail):
        keep = True                                   # consistent, keep as is
    elif boxed is None and tail:
        body += f"\nThe final answer is $\\boxed{{{tail}}}$."
        boxed, keep = tail, True                      # add the missing box
    else:
        keep = False                                  # contradictory or empty

    subj, level = math_lookup.get(norm(ex["original_question"]), (None, None))
    return {"problem": ex["query"], "solution": body, "answer": boxed,
            "subject": subj, "level": level, "source": ex["type"], "keep": keep}

mm_final = chk.map(finalize, remove_columns=chk.column_names)
print("keep counts:", Counter(mm_final["keep"]))
mm_final = mm_final.filter(lambda x: x["keep"]).remove_columns("keep")

# Sanity: every kept row's answer must be re-extractable from its solution
bad = sum(get_last_boxed(s) is None for s in mm_final["solution"])
print("Rows whose solution has no extractable box:", bad)   # expect 0

keep counts: Counter({True: 115669, False: 1279})
Rows whose solution has no extractable box: 0


# 8. Merge into one SFT dataset

In [21]:
# Put MATH train into the same schema
math_sft_std = math_sft.map(lambda x: {
    "problem": x["problem"], "solution": x["solution"], "answer": x["answer"],
    "subject": x["type"], "level": x["level"], "source": "MATH_train"},
    remove_columns=math_sft.column_names)

sft_all = concatenate_datasets([math_sft_std, mm_final]).shuffle(seed=SEED)
print("Total SFT examples:", len(sft_all))
print(Counter(sft_all["source"]))
print(Counter(sft_all["level"]))

Map:   0%|          | 0/6998 [00:00<?, ? examples/s]

Total SFT examples: 122667
Counter({'MATH_AnsAug': 69310, 'MATH_Rephrased': 46359, 'MATH_train': 6998})
Counter({'Level 2': 35152, 'Level 3': 31322, 'Level 4': 23215, 'Level 1': 18833, 'Level 5': 14108, 'Level ?': 37})


# 9. Decontamination check against MATH-500
- everything traces to MATH train so I created a heuristic verifier:
    - exact math on the normalized problem text 
    - 13 gram word overlap 

In [22]:
N = 13

def ngrams(text, n=N):
    words = re.sub(r"\s+", " ", text.lower()).split(" ")
    return {" ".join(words[i:i + n]) for i in range(len(words) - n + 1)}

test_problems = {norm(p) for p in math500["problem"]}
test_ngrams = set()
for p in math500["problem"]:
    test_ngrams |= ngrams(p)

exact_hits = [i for i, p in enumerate(sft_all["problem"]) if norm(p) in test_problems]
ngram_hits = [i for i, p in enumerate(sft_all["problem"]) if ngrams(p) & test_ngrams]

print(f"Exact matches with MATH-500: {len(exact_hits)}")
print(f"{N}-gram overlaps with MATH-500: {len(ngram_hits)}")

Exact matches with MATH-500: 0
13-gram overlaps with MATH-500: 795


In [23]:
# Inspect any n-gram hits. Shared boilerplate ("Express your answer as a common fraction")
# is a false positive; a near-copy of a test problem is a real hit and must be removed.
for i in ngram_hits[:5]:
    p = sft_all[i]["problem"]
    shared = ngrams(p) & test_ngrams
    print(sft_all[i]["source"], "|", p[:200])
    print("  shared:", list(shared)[:2], "\n")

MATH_AnsAug | In the diagram, $D$ and $E$ are the midpoints of $\overline{AB}$ and $\overline{BC}$ respectively. Find the sum of the slope and $y$-intercept of the line passing through the points $C$ and $D.$ [asy]
  shared: ['label("$a(0,6)$", a, w); label("$b(0,0)$", b, sw); label("$c(8,0)$", c, s); label("$d$", d, w); label("$e$",', 'c=(8,0); d=(0,3); e=(4,0); f=(8/3,2); draw(e--a--c--d); draw((-1,0)--(10,0), endarrow); draw((0,-1)--(0,8), endarrow); label("$a(0,6)$", a, w); label("$b(0,0)$",'] 

MATH_AnsAug | Let $a,$ $b,$ and $c$ be positive real numbers.  Find the minimum value of
\[\frac{a + b}{c} + \frac{a + c}{b} + \frac{b + c}{a}.\]
  shared: ['$a,$ $b,$ and $c$ be positive real numbers. find the minimum value of', 'let $a,$ $b,$ and $c$ be positive real numbers. find the minimum value'] 

MATH_AnsAug | In polar coordinates, the point $\left( -2, \frac{3 \pi}{8} \right)$ is equivalent to what other point, in the standard polar coordinate representation?  Enter your answer in 

In [24]:
# Remove real hits (exact matches always; n-gram hits only after inspection above)
REMOVE = set(exact_hits)          # add confirmed n-gram hits here if any
if REMOVE:
    sft_all = sft_all.select([i for i in range(len(sft_all)) if i not in REMOVE])
print("Final SFT size:", len(sft_all))

Final SFT size: 122667


# 10. Save/Summary

In [20]:
sft_all.save_to_disk("data/sft_all")
math_val.save_to_disk("data/math_val")
math500.save_to_disk("data/math500")
print("Saved to data/")

Saving the dataset (0/1 shards):   0%|          | 0/122667 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 500/500 [00:00<00:00, 20586.35 examples/s]

Saved to data/


In [25]:
summary = {
    "MATH train raw": 7500,
    "MATH train failed extraction (dropped)": len(failed),
    "MATH SFT split": len(math_sft),
    "MATH val split": len(math_val),
    "MetaMathQA MATH_* rows": len(metamath_math),
    "MetaMathQA match rate to MATH train": f"{match_rate:.1%}",
    "MetaMathQA rows dropped (val leak)": len(metamath_math) - len(mm),
    "MetaMathQA types dropped": sorted(DROP_TYPES),
    "MetaMathQA rows kept after consistency": len(mm_final),
    "MATH-500 exact matches": len(exact_hits),
    "MATH-500 13-gram overlaps": len(ngram_hits),
    "Final SFT size": len(sft_all),
}
for k, v in summary.items():
    print(f"{k:45s} {v}")

MATH train raw                                7500
MATH train failed extraction (dropped)        2
MATH SFT split                                6998
MATH val split                                502
MetaMathQA MATH_* rows                        155000
MetaMathQA match rate to MATH train           100.0%
MetaMathQA rows dropped (val leak)            10047
MetaMathQA types dropped                      ['MATH_FOBAR', 'MATH_SV']
MetaMathQA rows kept after consistency        115669
MATH-500 exact matches                        0
MATH-500 13-gram overlaps                     795
Final SFT size                                122667


In [26]:
fobar = chk.filter(lambda x: x["type"] == "MATH_FOBAR").shuffle(seed=0).select(range(20))
for i, ex in enumerate(fobar):
    print(f"[{i}] LABEL: {ex['tail']}")
    print(ex["response"][-350:], "\n")

# 12.Prompt format 

In [27]:
from posttrain.prompts import to_prompt_completion

# this is the format we want the model to follow

sft_all = sft_all.map(to_prompt_completion)
print(sft_all[0]['prompt'] + sft_all[0]['completion'])

Map:   0%|          | 0/122667 [00:00<?, ? examples/s]

<|im_start|>system
Please reason step by step, and put your final answer within \boxed{}.<|im_end|>
<|im_start|>user
Between 50 and 250, how many numbers are perfect squares?<|im_end|>
<|im_start|>assistant
The largest perfect square less than 250 is $15^2=225$,
and the smallest perfect square greater than 50 is $8^2=64$.
So, the perfect squares between 50 and 250 are $64, 81, 100, 121, 144, 169, 196, 225$,
which is a total of $\boxed{8}$ numbers.<|im_end|>


# 13.Token lengths

In [26]:
%pip install transformers --break-system-packages

  Using cached huggingface_hub-2.4.0-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.2 kB)
  Using cached click-8.5.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.8.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached httpx2-2.16.0-py3-none-any.whl.metadata (10 kB)
  Using cached httpcore2-2.16.0-py3-none-any.whl.metadata (27 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached truststore-0.10.5-py3-none-any.whl.metadata (4.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 5.1 MB/s  0:00:02 eta 0:00:01
Using cached click-8.5.1-py3-none-any.whl (126 kB)
Using cached hf_xet-1.8.0-cp38-abi3-macosx_11_0_arm64.whl (3.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 4.6 MB/s  0:00:00
Using cached idna-3.20-py3-none-any.whl (69 kB)
Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl (484 kB)
  Attempting uninstall: idna━━━━━━━━━━━━━━━━━

In [28]:
# we need to download the tokenizer from the qwen model

from transformers import AutoTokenizer 
import numpy as np 

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B-Base")

def count_tokens(batch):
    full = [p + c for p,c in zip(batch['prompt'], batch['completion'])]
    return {'n_tokens': [len(ids) for ids in tokenizer(full)['input_ids']]}


sft_all = sft_all.map(count_tokens, batched = True)

lens = np.array(sft_all['n_tokens'])
for p in [50, 90, 95, 99.9]:
    print(f'p{p}: {np.percentile(lens, p)} tokens')
print('max:', lens.max())


for limit in [512, 768, 1024, 1536, 2048]:
    print(f"<= {limit}: {(lens <= limit).mean():.2%}")

Map:   0%|          | 0/122667 [00:00<?, ? examples/s]

p50: 201.0 tokens
p90: 361.0 tokens
p95: 445.0 tokens
p99.9: 1184.6680000000342 tokens
max: 2747
<= 512: 96.88%
<= 768: 99.20%
<= 1024: 99.80%
<= 1536: 99.97%
<= 2048: 99.99%


- in the code above we use three functions from prompts.py the build_prompt(problem) which wraps the question in the fixed template which is system instructions, then the user turn with the problem, then the opening of the assistant turn
    - build_completion(solution) is the solution text plus <|im_end|> so the model knows where to stop 

- in the section above we tokenize prompt + completion together and count the tokens.
    - we do this because in SFT the model reads the whole sequence but the loss is computed only on the completion but the prompt still takes up space in the sequence and in GPU memory.

In [29]:
MAX_LEN = 1024   # adjust after looking at the percentiles above
before = len(sft_all)
sft_all = sft_all.filter(lambda x: x["n_tokens"] <= MAX_LEN)
print(f"Dropped {before - len(sft_all)} examples over {MAX_LEN} tokens; {len(sft_all)} remain")

sft_all.save_to_disk("data/sft_all")

Filter:   0%|          | 0/122667 [00:00<?, ? examples/s]

Dropped 251 examples over 1024 tokens; 122416 remain


Saving the dataset (0/1 shards):   0%|          | 0/122416 [00:00<?, ? examples/s]